In [ ]:
# Imports

import torch
import torch.nn as nn
import torch.optim as optim
import ta
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [2]:
# Load data
# yf.download downloads("Stock name", start="Date you want it from")
df = yf.download("QQQ", start="2009-12-01")

[*********************100%***********************]  1 of 1 completed


In [3]:
# Exploring Data
# Check for missing values
df.isnull().sum()

# Using shift -1, and then check if current day compare to tomorrow is higher or lower based on Close values, 1 = True, 0 = False
# From 2020-01-02 The closing value was 208, taking the next day's closing value, 2020-01-03, the closing value is 206 so it asks if 206 > 208 = F
# Then it would mark 2020-01-02 as 0 since the next day was not higher than today's. 

# Essentially asking, is tomorrow closing vlaue going to be higher?
df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

# Dropped Weekly, Monthly MAs, and RSI
# # Make a new Column to find out the past 7 day average, .rolling uses the last (n), in this case, 7 rows before it. 
# df["Weekly_MA"] = df["Close"].rolling(7).mean()

# # Make a new Column to find out the past 30 day average
# df["Monthly_MA"] = df["Close"].rolling(30).mean().squeeze()

# Change in volume
df["Volume_Change"] = df["Volume"].pct_change() * 100

# RSI
# df["RSI"] = ta.momentum.RSIIndicator(df["Close"].squeeze()).rsi()
# df["RSI"].tail()

# EMAs
df["EMA_9"] = ta.trend.EMAIndicator(df["Close"].squeeze(), window=9).ema_indicator()
df["EMA_21"] = ta.trend.EMAIndicator(df["Close"].squeeze(), window=21).ema_indicator()
df["EMA_50"] = ta.trend.EMAIndicator(df["Close"].squeeze(), window=50).ema_indicator()
df["EMA_200"] = ta.trend.EMAIndicator(df["Close"].squeeze(), window=200).ema_indicator()


# Automatically removes NaN rows 
df = df.dropna()
print(df.shape)
df.tail()

(3930, 11)


Price,Close,High,Low,Open,Volume,Target,Volume_Change,EMA_9,EMA_21,EMA_50,EMA_200
Ticker,QQQ,QQQ,QQQ,QQQ,QQQ,,,,,,
Date,,,,,,,,,,,
2026-04-27,664.229980,664.429993,660.690002,663.400024,32717000,0,-28.194087,648.301580,629.369469,615.621550,592.373362
2026-04-28,657.549988,659.640015,653.809998,657.409973,34147900,1,4.373567,650.151261,631.931334,617.265802,593.021885
2026-04-29,661.570007,661.719971,656.590027,658.630005,31724900,1,-7.095605,652.435011,634.625759,619.003222,593.703956
2026-04-30,667.739990,668.900024,657.559998,665.349976,40622200,1,28.045163,655.496007,637.636143,620.914468,594.440633
2026-05-01,674.150024,675.969971,668.799988,669.159973,39055400,0,-3.857004,659.226810,640.955587,623.002137,595.233762


## Structuring

Input (features)

    ↓

Linear Layer (9 -> 64)

    ↓

ReLU activation

    ↓

Linear Layer (64 -> 32)

    ↓

ReLU activation

    ↓

Linear layer (32 ->2) <- 2 outputs (UP OR DOWN)


In [4]:

# Testing and Training data
X = df.drop(columns=["Target"])
y = df["Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=250, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy_score(y_test, predictions)


0.5216284987277354

In [5]:
# Scale the data
scaler = StandardScaler()
Xtrain = scaler.fit_transform(X_train)
Xtest = scaler.transform(X_test)

# Convert to tensors
Xtrain = torch.tensor(Xtrain, dtype=torch.float32)
Xtest = torch.tensor(Xtest, dtype=torch.float32)

Ytrain = torch.tensor(y_train.values, dtype=torch.long)
Ytest = torch.tensor(y_test.values, dtype=torch.long)


In [6]:
# Building the NN
# Takes a list of layers, passes input through each layer one at a time, output of one layer becomes input of the next
# Input -> Layer 1 -> Layer 2 -> Layer 3 -> Output
torch.manual_seed(42) # Make sure its consistent

model_nn = nn.Sequential(
    nn.Linear(10, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

In [7]:
# Loss Function
# CrossEntropyLoss: measures the performance of classifcation model by calculating the difference between predicted probability distributions and actual labels
# softmax -> log -> nll = CrossEntropyLoss
criterion = nn.CrossEntropyLoss()

# Optimization
# optim.Adam is gradient descent
# p.data += -lr * p.grad
# Reminder: Gradient descent is how the model learns by adjusting its weight to reduce the loss
optimizer = optim.Adam(model_nn.parameters(), lr=0.001)

In [8]:
# Training loop
torch.manual_seed(42) # Make sure its consistent

# adding batches
max_steps = 5000
batch_size = 32

for epoch in range(max_steps):
    # random minibatch
    ix = torch.randint(0, Xtrain.shape[0], (batch_size,))
    Xb = Xtrain[ix]
    Yb = Ytrain[ix]

    # forward pass
    logits = model_nn(Xb)
    loss = criterion(logits, Yb)

    # backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # # Learning rate decay 
    # lr = 0.001 if epoch < 15000 else 0.0001
    # for param_group in optimizer.param_groups:
    #     param_group['lr'] = lr

    # Update
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

Epoch 0: Loss = 0.7186
Epoch 1000: Loss = 0.7039
Epoch 2000: Loss = 0.6814
Epoch 3000: Loss = 0.6696
Epoch 4000: Loss = 0.6477


In [9]:
with torch.no_grad():
    test_logits = model_nn(Xtest)
    predictions_nn = test_logits.argmax(dim=1)
    accuracy_nn = (predictions_nn == Ytest).float().mean()
    print(f"NN: {accuracy_nn:.2%}")

NN: 56.49%
